# Reconstruction Impact：用三个问题阅读 Raw/SVC

本 Notebook 是针对一个已声明 Raw/SVC 样本的可读执行路径。计算阶段仍按 `STAGE_ORDER` 展开，并与批量运行器和静态报告使用同一套已保存产物协议。

科学阅读按三个问题推进：

1. **重建后呈现什么，两侧差异在哪里？** 先读输入提供的 SVC 重建状态标签，再看 Raw 独立表达 baseline 和两侧各自的空间结果。
2. **State 与差异出现在哪里？** 先看 SVC 标签多样性和共同有效窗口上的 ΔNeff，再把这些窗口放回 SVC Anatomy；State 与 Gain 是不同的证据视图。
3. **位置与分子/成员有什么关系？** Moran 是两侧原生空间图上的平行分支；AUCell 先在固定单位和基因轴上评分，再做空间聚合；membership 只比较明确的共同 ID；集成表只连接已保存事实，不构成独立验证。

输入的 SVC 重建标签就是重建状态标签。每个阶段按自身所需的字段、表达身份和空间坐标判断 capability；缺少某一前提只使依赖它的阶段标为 `unavailable`/`partial`，不会静默替换成新的 Leiden、备用 fixture 或零值。样本声明缺失或无效时会在加载阶段直接报错。


## 输入与参数协议

无环境变量时默认使用 `configs/p2_project.yaml`，并从项目声明的唯一 sample YAML 推导 `SAMPLE_YAML`。显式设置 `SAMPLE_YAML` 时，如果没有同时显式设置 `PROJECT_YAML`，则不套用默认 project；若同时设置两者，仍按 project 的 `samples` 做 membership 校验。显式 `PROJECT_YAML` 未配 `SAMPLE_YAML` 时，只有单样本项目可以自动推导；多样本项目必须显式选择样本。`OUTPUT_DIR` 更换输出位置，`RECONSTRUCTION_IMPACT_OVERRIDES` 接受本次交互覆盖的 JSON 映射。参数优先级是包默认值 < 样本 YAML < 项目 YAML < Notebook override。

Raw `.X` 是上游交付的原始侧矩阵，SVC `.X` 是重建侧矩阵；本仓库只核验声明和消费前提，不改写任一 H5AD。正式 `.X` 契约固定为 finite、nonnegative、unlogged linear，允许小数和小于 1 的值；旧 `log`/`log1p` 声明会被拒绝。`identity: unknown` 仍可运行标签/空间阶段，但不能据此放行 P2 的表达分析。


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

REPO_ROOT = next(
    candidate for candidate in (Path.cwd(), Path.cwd().parent)
    if (candidate / "revise_analysis").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from IPython.display import Image, Markdown, display
except Exception:
    Image = None
    Markdown = None
    display = print

from revise_analysis.io import load_sample, read_yaml
from revise_analysis.runner import resolve_analysis_parameters
from revise_analysis.analyses.reconstruction_impact import ImpactWorkflow, STAGE_ORDER, effective_parameters

def _repo_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO_ROOT / path).resolve()

def _project_sample(project_yaml, project_config):
    project_samples = list(project_config.get("samples", []) or [])
    if len(project_samples) != 1:
        raise ValueError("多样本项目必须显式设置 SAMPLE_YAML；Notebook 不会静默选择。")
    sample_path = Path(project_samples[0]).expanduser()
    return (sample_path if sample_path.is_absolute() else project_yaml.parent / sample_path).resolve()

DEFAULT_PROJECT = REPO_ROOT / "configs" / "p2_project.yaml"
PROJECT_TEXT = os.environ.get("PROJECT_YAML", "").strip()
SAMPLE_TEXT = os.environ.get("SAMPLE_YAML", "").strip()

if SAMPLE_TEXT:
    SAMPLE_YAML = _repo_path(SAMPLE_TEXT)
    # An explicit sample alone must not inherit the default project.
    PROJECT_YAML = _repo_path(PROJECT_TEXT) if PROJECT_TEXT else None
else:
    PROJECT_YAML = _repo_path(PROJECT_TEXT) if PROJECT_TEXT else DEFAULT_PROJECT.resolve()
    PROJECT_CONFIG = read_yaml(PROJECT_YAML)
    SAMPLE_YAML = _project_sample(PROJECT_YAML, PROJECT_CONFIG)

PROJECT_CONFIG = read_yaml(PROJECT_YAML) if PROJECT_YAML else {}

OVERRIDE_TEXT = os.environ.get("RECONSTRUCTION_IMPACT_OVERRIDES", "{}")
OVERRIDES = json.loads(OVERRIDE_TEXT)
if not isinstance(OVERRIDES, dict):
    raise ValueError("RECONSTRUCTION_IMPACT_OVERRIDES must be a JSON object")

display(pd.DataFrame([
    {"参数": "SAMPLE_YAML", "值": str(SAMPLE_YAML)},
    {"参数": "PROJECT_YAML", "值": str(PROJECT_YAML) if PROJECT_YAML else "<未设置>"},
    {"参数": "OUTPUT_DIR", "值": os.environ.get("OUTPUT_DIR", "<加载样本后推导>")},
    {"参数": "RECONSTRUCTION_IMPACT_OVERRIDES", "值": OVERRIDE_TEXT},
    {"参数": "continue_on_error", "值": False},
]))


## 加载并验证已声明样本

加载只验证 sample.yaml、文件身份和可读性；各阶段按自身需要的字段、表达身份和坐标前提判断 capability。缺少 SVC reconstruction 字段不会被新的 Leiden、备用 fixture 或零值替换；只影响依赖该字段的 State/Gain、membership 或其他对应阶段，Raw/SVC 的独立空间能力按各自输入继续判断。


In [ ]:
sample = load_sample(SAMPLE_YAML)

DEFAULT_OUTPUT = REPO_ROOT / "output" / "notebook" / sample.sample_id
OUTPUT_DIR = _repo_path(os.environ.get("OUTPUT_DIR", str(DEFAULT_OUTPUT)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PARAMETERS = dict(
    sample.config.get("analysis_parameters", {}).get("reconstruction_impact", {}) or {}
)
PROJECT_PARAMETERS = dict(
    PROJECT_CONFIG.get("analyses", {}).get("reconstruction_impact", {}) or {}
)

def resolve_current_parameters():
    explicit = resolve_analysis_parameters(
        SAMPLE_YAML,
        "reconstruction_impact",
        project_yaml=PROJECT_YAML,
        overrides=OVERRIDES,
    )
    return explicit, effective_parameters(sample, explicit)

workflow = ImpactWorkflow(sample, OUTPUT_DIR, continue_on_error=False)
display(Markdown(
    f"已加载 {sample.sample_id!r}；SVC 重建标签声明为 {sample.reconstruction_key!r}，各阶段将按实际字段能力独立判断。"
))


## 应用参数并查看失效阶段

修改 `OVERRIDES` 后重新运行本单元。它会重新读取样本/项目参数、合并 Notebook override，并把最终参数显式应用到工作流。已完成但受参数影响的阶段回到 `pending`；无关阶段保持有效。阈值参数变化不会要求重算 Moran 或 AUCell。


In [ ]:
EXPLICIT_PARAMETERS, EFFECTIVE_PARAMETERS = resolve_current_parameters()
# apply_parameters 接受完整最终参数快照；删除 override 时才能恢复配置/default 值。
invalidated = workflow.apply_parameters(EFFECTIVE_PARAMETERS)
result = workflow.result()

parameter_rows = []
for name, value in EFFECTIVE_PARAMETERS.items():
    source = "包默认值"
    if name in SAMPLE_PARAMETERS:
        source = "sample.yaml"
    if name in PROJECT_PARAMETERS:
        source = "project.yaml"
    if name in OVERRIDES:
        source = "Notebook override"
    parameter_rows.append({
        "参数": name,
        "sample.yaml": SAMPLE_PARAMETERS.get(name, ""),
        "project.yaml": PROJECT_PARAMETERS.get(name, ""),
        "Notebook override": OVERRIDES.get(name, ""),
        "最终来源": source,
        "最终值": value,
    })

display(Markdown("失效阶段：" + (", ".join(invalidated) if invalidated else "无；当前结果仍适用")))
display(pd.DataFrame(parameter_rows))
display(pd.DataFrame(result.get("stages", [])))


## 执行有序阶段

下面每个 Notebook 小节都通过 `run_stage` 调用一个命名的 `ImpactWorkflow` 阶段，然后读取当前结果清单。执行顺序保留为：输入、baseline、支持诊断、多样性、区域、Anatomy、分子结果、membership、集成和已保存结果图。三个问题只改变阅读转换，不改变阶段顺序、有效参数或计算对象。意外错误会继续传播；缺少科学前提会明确保留在当前结果中。


In [ ]:
result = workflow.result()
display(pd.DataFrame([
    {"顺序": index, "阶段": name}
    for index, name in enumerate(STAGE_ORDER, start=1)
]))
display(pd.DataFrame(result.get("stages", [])))


## 按阶段阅读已保存产物

下面的辅助函数只读取刚刚保存到 `OUTPUT_DIR` 下、且列在当前 `result.outputs` 中的文件。这个 manifest 是 Notebook 的产物边界：目录里的旧文件不会被猜测为当前结果。当某个阶段只有部分结果时，Notebook 仍能保持可读，并为每个小节显示对应的产物路径。


In [ ]:
STAGE_PARAMETER_KEYS = {
    "input": (),
    "baseline": ("resolution", "n_top_genes", "sample_n_units", "random_state", "svc_leiden_baseline"),
    "support": ("scopes", "parent_window_side_microns", "window_scale_candidates_microns", "min_window_units"),
    "diversity": ("scopes", "parent_window_side_microns", "min_window_units", "n_window_draws", "random_state"),
    "regions": ("region_n_bootstrap",),
    "anatomy": ("anatomy_window_side_microns", "anatomy_tumor_label", "anatomy_normal_label"),
    "molecular": ("sample_n_units", "moran_n_neighbors", "pathway_auc_threshold", "gene_set_names"),
    "membership": ("membership_same_units",),
    "integration": ("scopes",),
}

def saved_path(relative):
    manifest = set(result.get("outputs", {}).values())
    if relative not in manifest:
        return None
    path = OUTPUT_DIR / relative
    return path if path.is_file() else None

def show_saved_table(relative, rows=12):
    path = saved_path(relative)
    if path is None:
        return False
    print(relative)
    if path.suffix.lower() == ".csv":
        display(pd.read_csv(path).head(rows))
    elif path.suffix.lower() == ".json":
        display(pd.DataFrame([json.loads(path.read_text(encoding="utf-8"))]))
    elif path.suffix.lower() in {".png", ".jpg", ".jpeg", ".svg"} and Image is not None:
        display(Image(filename=str(path)))
    else:
        print(f"已保存文件：{path}")
    return True

def show_section(section_id, rows=12):
    section = next(
        (item for item in result.get("sections", []) if item.get("id") == section_id),
        {"id": section_id, "artifacts": []},
    )
    display(Markdown(f"### {section.get('title', section_id)}"))
    if section.get("description"):
        display(Markdown(section["description"]))
    keys = STAGE_PARAMETER_KEYS.get(section_id, ())
    if keys:
        display(pd.DataFrame([{"本节关键参数": key, "值": workflow.parameters.get(key)} for key in keys]))
    artifacts = section.get("artifacts", [])
    for relative in artifacts:
        show_saved_table(relative, rows=rows)
    if not artifacts:
        display(Markdown("本阶段没有保存产物。"))

def run_visible_stage(name):
    global result
    current_explicit, pending = resolve_current_parameters()
    if pending != workflow.parameters:
        raise RuntimeError(
            "样本、项目或 Notebook override 已改变。请先重新运行“应用参数并查看失效阶段”单元，"
            "确认失效范围后再执行阶段。"
        )
    record = workflow.run_stage(name)
    workflow.render_figures(name)
    result = workflow.result()
    display(pd.DataFrame([{
        "阶段": record.get("stage"),
        "状态": record.get("status"),
        "产物": ", ".join(record.get("artifacts", [])),
    }]))
    show_section(name)
    return record


In [ ]:
def show_section(section_id, rows=12, max_tables=4):
    section = next(
        (item for item in result.get("sections", []) if item.get("id") == section_id),
        {"id": section_id, "artifacts": []},
    )
    display(Markdown(f"### {section.get('title', section_id)}"))
    if section.get("description"):
        display(Markdown(section["description"]))
    keys = STAGE_PARAMETER_KEYS.get(section_id, ())
    if keys:
        display(pd.DataFrame([{"本节关键参数": key, "值": workflow.parameters.get(key)} for key in keys]))
    artifacts = list(section.get("artifacts", []))
    figures = [path for path in artifacts if Path(path).suffix.lower() in {".png", ".jpg", ".jpeg", ".svg"}]
    tables = [path for path in artifacts if path not in figures]
    for relative in figures:
        show_saved_table(relative, rows=rows)
    for relative in tables[:max_tables]:
        show_saved_table(relative, rows=rows)
    if len(tables) > max_tables:
        display(Markdown(f"其余 {len(tables) - max_tables} 份登记底表保留在输出 manifest 与最终文件索引中。"))
    if not artifacts:
        display(Markdown("本阶段没有保存产物。"))


## 问题一：重建后呈现什么，两侧差异在哪里？

输入小节记录单位数、基因数以及输入的重建状态标签计数；这些标签是 State 的对象，不由 Notebook 重新命名。baseline 小节分别标出 Raw Leiden 和已有 Raw Level2 证据。支持曲线先说明候选尺度下的 SVC parent、Raw 和共同有效窗口支持，但不使用 State、Gain 或 program 结果反向选择尺度。


In [ ]:
run_visible_stage("input")


In [ ]:
run_visible_stage("baseline")


In [ ]:
run_visible_stage("support")


## 问题二：State 与差异出现在哪里？

State 直接读取已保存的 SVC 重建状态标签列。窗口多样性表会在阈值判断之前展示，因此即使结果为 `no_stable_threshold`，连续场也不会被删除。下一步的 Gain 只在共同有效窗口中读取 SVC 与 Raw Leiden 的 ΔNeff；它不能把标签变化单独解释成生物学改善。


In [ ]:
run_visible_stage("diversity")


## 问题二续：把差异放回空间与 Anatomy

State 区域是连续 SVC 多样性场经过阈值筛选后的视图。Gain 是共同有效窗口上相对于 Raw Leiden 的候选 ΔNeff；两者的支持、分母和阈值状态分别保留。阈值失败时仍保留连续表、阈值状态和 bootstrap 审计，不会把结果伪装成数值为零的区域。


In [ ]:
run_visible_stage("regions")


## 问题二续：Anatomy 是空间解释背景

SVC Anatomy 由 SVC broad 标签与坐标定义；Raw 复用同一 shared origin 进行空间落点，Raw 点没有对应的 SVC Anatomy 网格时标为 `Unknown`。`Other` 只表示当前窗口没有配置的 Tumor 或 Normal 来源，不表示原始 tissue 不存在 Tumor。parent 窗口保留实际观测点组成，包括并列主导和缺失覆盖；不使用 parent 中心点或重叠面积的捷径。Anatomy 解释 State/Gain 的位置，但不替代 State/Gain 的定义。


In [ ]:
run_visible_stage("anatomy")


## 问题三：位置与分子/成员有什么关系？

Moran 是与 State/Gain 平行的分支：Raw 与 SVC 各自在自己的原生坐标图上计算，不依赖 Region，也不因为某个 Region 再建图。AUCell 先对固定单位集、基因轴和 gene set 做一次单位级评分，再将已保存分数按 window、SVC Anatomy 或 Region 聚合；聚合不会重复评分。只有显式开启共同 ID 选项时才会生成 membership，且只比较 Raw-defined cohort 与实际可用 SVC 标签的 shared IDs。集成结果把已保存的 State/Gain、SVC Anatomy、分子和 membership 事实按明确分母连接起来；它不会把部分完成的阶段提升为独立验证。


In [ ]:
run_visible_stage("molecular")


In [ ]:
run_visible_stage("membership")


In [ ]:
run_visible_stage("integration")


## 三个问题的限制与最终综合

最后只保存当前工作流清单、渲染静态报告并从已保存事实形成综合阅读。它不会补跑 `pending` 科学阶段、重新打开输入对象或新增评分。若参数变更使阶段失效，应回到对应小节重跑后再保存。


In [ ]:
# 发布前再次确认参数已经显式应用；随后只刷新登记图，不运行 scientific stage。
_, pending_parameters = resolve_current_parameters()
if pending_parameters != workflow.parameters:
    raise RuntimeError("参数来源已改变；请先重新运行“应用参数并查看失效阶段”单元。")
workflow.run_stage("figures")


In [ ]:
result = workflow.result()
pending_stages = [row["stage"] for row in result.get("stages", []) if row.get("status") == "pending"]
if pending_stages:
    display(Markdown("### 仍为 pending 的科学阶段\n\n" + ", ".join(pending_stages)))

def _json_safe(value):
    if isinstance(value, dict):
        return {str(key): _json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "item"):
        return _json_safe(value.item())
    return value

# 保存当前清单；本单元不补跑任何 scientific stage。
result_for_report = dict(result)
result_for_report.update(schema_version=1, sample_id=sample.sample_id, analysis="reconstruction_impact")
result_for_report["outputs"] = dict(result.get("outputs", {}))
result_for_report["outputs"]["report"] = "report.html"
(OUTPUT_DIR / "result.json").write_text(
    json.dumps(_json_safe(result_for_report), ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
from revise_analysis.reporting import render_report
from revise_analysis.reporting.impact_reading import reading_summary
report_path = render_report(OUTPUT_DIR)

# 综合事实只读取同一份已保存结果，不触发任何科学阶段。
reading_items = reading_summary(OUTPUT_DIR, result_for_report)[:5]
reading_lines = []
for item in reading_items:
    text = str(item.get("text", "")).strip()
    anchor = str(item.get("anchor", "")).strip().lstrip("#")
    if not text:
        continue
    target = f"report.html#{anchor}" if anchor else "report.html"
    reading_lines.append(f"- {text} ([报告定位]({target}))")
if reading_lines:
    display(Markdown("### 三个问题的综合阅读\n\n" + "\n".join(reading_lines)))

unavailable_rows = result.get("unavailable", [])
error_rows = result.get("stage_errors", [])
if unavailable_rows:
    display(Markdown("### 当前不可用组件"))
    display(pd.DataFrame(unavailable_rows))
if error_rows:
    display(Markdown("### 阶段执行错误"))
    display(pd.DataFrame(error_rows)[
        [column for column in ("stage", "error_type", "message")
         if column in pd.DataFrame(error_rows).columns]
    ])
file_index = pd.DataFrame([
    {"输出键": key, "保存路径": relative}
    for key, relative in sorted(result_for_report["outputs"].items())
])
display(Markdown(f"静态报告已保存到：[{report_path.name}]({report_path.name})"))
display(Markdown("### 已保存文件索引"))
display(file_index)
